# calibra-ia-api — Fase 1: Preparação e Serialização do Modelo

**Origem do modelo (Opção A):** reaproveitamento do mini-projeto `calibra-ia` (classificação de dígitos MNIST), no qual o **MLP** obteve o melhor desempenho entre os três modelos avaliados (F1 ponderado = 0,9808).

**Objetivo desta fase:** retreinar o modelo dentro de um `Pipeline` do scikit-learn, com divisão treino/validação/teste estratificada, e serializá-lo em `models/` para consumo pela API FastAPI.

In [1]:
import sys
import numpy as np
import sklearn
import joblib
import fastapi
import pydantic

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"joblib      : {joblib.__version__}")
print(f"FastAPI     : {fastapi.__version__}")
print(f"Pydantic    : {pydantic.__version__}")

Python      : 3.14.5
NumPy       : 2.5.4
scikit-learn: 1.9.1
joblib      : 1.6.0
FastAPI     : 0.162.0
Pydantic    : 2.14.2


## 1. Carregamento dos dados

Dataset `mnist_784` (OpenML): 70.000 imagens 28×28 de dígitos manuscritos, achatadas em 784 pixels com intensidade de 0 a 255. O download ocorre uma única vez; as execuções seguintes leem o cache local em `data/mnist/`, que fica fora do versionamento pelo `.gitignore`.

In [ ]:
import time
from pathlib import Path
from sklearn.datasets import fetch_openml

RANDOM_STATE = 42

CACHE_DIR = Path("../data/mnist")          # caminho relativo, como exige o edital
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / "mnist_784.npz"

t0 = time.perf_counter()
if CACHE_FILE.exists():
    dados = np.load(CACHE_FILE)
    X, y = dados["X"], dados["y"]
    origem = "cache local"
else:
    mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="liac-arff")
    X = mnist.data.astype(np.float32)
    y = mnist.target.astype(np.uint8)
    np.savez_compressed(CACHE_FILE, X=X, y=y)
    origem = "download OpenML"

print(f"Origem   : {origem} ({time.perf_counter() - t0:.1f}s)")
print(f"X        : {X.shape}  | min={X.min():.0f}  max={X.max():.0f}")
print(f"y        : {y.shape}  | classes={np.unique(y)}")

Origem   : cache local (0.4s)
X        : (70000, 784)  | min=0  max=255
y        : (70000,)  | classes=[0 1 2 3 4 5 6 7 8 9]


## 2. Divisão treino / validação / teste (70 / 15 / 15)

O edital exige três conjuntos. A divisão é feita em duas etapas encadeadas, ambas com `stratify` para preservar a proporção de cada dígito:

- **Treino (70%)**: ajuste dos parâmetros do modelo.
- **Validação (15%)**: acompanhamento do desempenho durante o desenvolvimento.
- **Teste (15%)**: avaliação final, usado uma única vez, sem influenciar decisões de modelagem.

In [4]:
from sklearn.model_selection import train_test_split

# Etapa 1: separa 70% para treino e 30% para o restante
X_treino, X_temp, y_treino, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

# Etapa 2: divide o restante ao meio (15% validação, 15% teste)
X_val, X_teste, y_val, y_teste = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

for nome, conj in [("Treino", y_treino), ("Validação", y_val), ("Teste", y_teste)]:
    print(f"{nome:<10}: {len(conj):>6,} amostras ({len(conj)/len(y):.0%})")

Treino    : 49,000 amostras (70%)
Validação : 10,500 amostras (15%)
Teste     : 10,500 amostras (15%)


## 3. Pré-processamento e Pipeline

O edital cita três tratamentos: imputação de nulos, codificação categórica e escalonamento. Para o MNIST:

- **Imputação de nulos: não se aplica.** A verificação abaixo confirma zero valores ausentes.
- **Codificação categórica: não se aplica.** As 784 features são intensidades numéricas de pixel.
- **Escalonamento: aplicado.** O `MinMaxScaler` leva os pixels de [0, 255] para [0, 1], o que estabiliza o gradiente da MLP. Com `clip=True`, valores acima do máximo visto no treino são limitados a 1, protegendo a API contra entradas atípicas.

O escalonador fica **dentro** do `Pipeline`: é ajustado apenas no treino (sem vazamento de dados) e é serializado junto com o modelo. Assim, a API recebe pixels brutos e não precisa replicar nenhuma transformação. Foi usado um transformador nativo do scikit-learn, e não uma função `lambda`, porque funções desse tipo não podem ser serializadas pelo `joblib`.

Os hiperparâmetros da MLP são os mesmos do `calibra-ia`, com uma exceção: `max_iter` passou de 30 para 60. Com o treino reduzido a 70% dos dados, o limite de 30 épocas interrompia o ajuste antes da convergência. O teto maior permite que o `early_stopping` (5 épocas sem melhora na validação interna) determine o ponto de parada.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPClassifier

print(f"Valores nulos em X: {np.isnan(X).sum()}")

pipeline = Pipeline([
    ("escala", MinMaxScaler(clip=True)),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(256, 128),
        learning_rate_init=0.001,
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=128,
        max_iter=60,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=RANDOM_STATE,
    )),
])

t0 = time.perf_counter()
pipeline.fit(X_treino, y_treino)
print(f"Treino concluído em {time.perf_counter() - t0:.1f}s "
      f"| épocas: {pipeline.named_steps['mlp'].n_iter_}")

Valores nulos em X: 0
Treino concluído em 100.4s | épocas: 31


## 4. Relatório de métricas

Métricas calculadas na validação e no teste. Como as classes do MNIST são praticamente balanceadas, a acurácia é informativa, mas o **F1 ponderado** é reportado como métrica principal por equilibrar precisão e recall entre os dez dígitos. A **ROC-AUC** multiclasse (estratégia *one-vs-rest*) mede a capacidade de separação a partir das probabilidades do modelo, as mesmas que a API devolverá como confiança.

In [7]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

def avaliar(nome, X_, y_):
    y_pred = pipeline.predict(X_)
    y_proba = pipeline.predict_proba(X_)
    return {
        "conjunto": nome,
        "acuracia": round(accuracy_score(y_, y_pred), 4),
        "f1_ponderado": round(f1_score(y_, y_pred, average="weighted"), 4),
        "roc_auc_ovr": round(roc_auc_score(y_, y_proba, multi_class="ovr"), 4),
    }

metricas = [avaliar("validacao", X_val, y_val), avaliar("teste", X_teste, y_teste)]
for m in metricas:
    print(m)

print("\nClassification report — TESTE")
print(classification_report(y_teste, pipeline.predict(X_teste), digits=4))

{'conjunto': 'validacao', 'acuracia': 0.9806, 'f1_ponderado': 0.9806, 'roc_auc_ovr': 0.9996}
{'conjunto': 'teste', 'acuracia': 0.9813, 'f1_ponderado': 0.9813, 'roc_auc_ovr': 0.9996}

Classification report — TESTE
              precision    recall  f1-score   support

           0     0.9809    0.9903    0.9856      1035
           1     0.9916    0.9949    0.9932      1181
           2     0.9773    0.9838    0.9805      1049
           3     0.9876    0.9655    0.9764      1071
           4     0.9842    0.9756    0.9799      1024
           5     0.9769    0.9810    0.9789       947
           6     0.9817    0.9903    0.9860      1032
           7     0.9843    0.9771    0.9807      1094
           8     0.9842    0.9736    0.9789      1024
           9     0.9632    0.9799    0.9715      1043

    accuracy                         0.9813     10500
   macro avg     0.9812    0.9812    0.9812     10500
weighted avg     0.9814    0.9813    0.9813     10500



## 5. Serialização do artefato

O `Pipeline` completo (escalonador + MLP) é salvo com `joblib` em `models/`. Um arquivo `metadata.json` registra versão do modelo, métricas e versões das bibliotecas, garantindo rastreabilidade. Por fim, o artefato é **recarregado do disco** e testado, comprovando que a serialização preserva o comportamento do modelo.

In [8]:
import json
import os

MODELS_DIR = Path("../models")
MODELO_PATH = MODELS_DIR / "mlp_mnist_v1.joblib"
META_PATH = MODELS_DIR / "metadata.json"

joblib.dump(pipeline, MODELO_PATH)

metadata = {
    "model_name": "mlp_mnist",
    "model_version": "v1",
    "origem": "calibra-ia (Opção A)",
    "n_features": 784,
    "classes": [int(c) for c in pipeline.classes_],
    "metricas": metricas,
    "versoes": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__},
}
META_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

# Prova de integridade: recarrega do disco e compara as predições
modelo_recarregado = joblib.load(MODELO_PATH)
iguais = (modelo_recarregado.predict(X_teste) == pipeline.predict(X_teste)).all()

print(f"Artefato : {MODELO_PATH} ({os.path.getsize(MODELO_PATH) / 1024**2:.2f} MB)")
print(f"Metadata : {META_PATH}")
print(f"Predições idênticas após recarga: {iguais}")

Artefato : ..\models\mlp_mnist_v1.joblib (2.71 MB)
Metadata : ..\models\metadata.json
Predições idênticas após recarga: True
